# Multi-Horizon Terminal Forecasting Benchmark

## Paper Narrative

The main story here is that **a prediction market should be treated not as an object for next-tick prediction, but as a noisy and strategic sensor of the future world**.

This benchmark answers the core paper question:
- if we inspect the market in advance, `1d`, `3d`, and `7d` before resolution,
- and separately at the early, middle, and late stages of the market lifecycle (`25%`, `50%`, `75%` life progress),
- can we recover the final event probability from the market state,
- and which signals help beyond the current market price itself.

For the paper, this is the central forecasting benchmark: it tests whether the model learns a useful world-state belief rather than just local price dynamics. An additional emphasis is that we **do not want to over-credit easy cases**, where the market is already near `95%/5%`, so we also inspect hard slices with non-saturated probabilities.

## Everyday Intuition: Why This Might Work

The intuition is simple:
- the current market price already aggregates a lot of information, but not perfectly;
- the market can lag, suffer from low liquidity, noise, and local distortions;
- the same time-to-resolution does not imply the same stage of the market lifecycle, so it is useful to look at both absolute time and life progress;
- if the model sees not only the current price, but also the path leading to it, activity, related market families, and external context, it may estimate the final outcome better in the states where the market has not fully converged yet.

## What Data We Will Use

- `markets`: market metadata, question text, dates, domain, and final outcome;
- `added_markets`: which markets were fully downloaded and how much history is available for them;
- `probabilities`: 5-minute `yes_probability` history and activity features;
- when available: structural, textual, and external covariates;
- additionally in the analysis stage: the duration of available history for each ticker, to understand whether the benchmark covers short-lived and long-lived markets.

## What Metrics To Track

- `log_loss` as the main probability-quality metric;
- `brier` as a complementary probabilistic metric;
- `roc_auc` as a sanity check;
- breakdowns across absolute horizons, life-progress horizons, and confidence slices;
- separately we track the hard-case slice `0.1 < p < 0.9`, where the model actually has room to improve over the market.

## What Models To Train

- `market_price` as the main trivial baseline;
- a simple supervised linear model;
- a stronger tree / boosting baseline;
- then ablations: what microstructure, structure, text, and external covariates contribute.

## How To Read This Notebook

This is not just an experiment with a few models, but a **research tutorial for a benchmark task**.

What happens here step by step:
1. We first formalize the terminal forecasting task and explain why it matters for the paper.
2. We then build a supervised dataset from the Polymarket SQLite tables.
3. Next we define the protocol: which splits, which metrics, and which anti-leakage constraints.
4. After that we compare several baseline models.
5. Finally, we interpret what good or bad results mean for the NeurIPS story.

## Task

**What we predict:**
- the final binary market outcome (`Yes/No`),
- when observing the market before resolution under two snapshot families:
  - absolute horizons: `1d`, `3d`, `7d` before resolution;
  - relative horizons: `25%`, `50%`, `75%` life progress.

**Input:**
- the market state at the snapshot time;
- the probability history up to that point;
- liquidity and activity features;
- when available: structural, textual, and external covariates.

**Target:**
- `target = 1` if the final outcome is `Yes`, otherwise `0`.

**Why this task matters:**
- it tests whether we can recover a useful belief state about the future event,
- rather than simply guess the next local price move;
- having two horizon families lets us separate the question “what does the model know at a fixed time before resolution?” from the question “what does the model know at different stages of the market lifecycle?”

In [1]:
# If needed in a fresh notebook environment:
# %pip install -r ../requirements.txt

from __future__ import annotations

import ast
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "benchmarks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "benchmarks"))

from benchmark_utils import (
    DEFAULT_DB_PATH,
    add_time_features,
    build_multi_horizon_terminal_dataset,
    build_repricing_dataset,
    connect,
    extract_snapshot_features,
    load_eligible_markets,
    load_probabilities_for_markets,
    rolling_time_splits,
)
from covariate_utils import (
    add_lagged_covariate_features,
    asof_join_covariates,
    load_external_covariates,
    pivot_covariates_to_wide,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DB_PATH = DEFAULT_DB_PATH
DOMAINS = ("geopolitics", "finance_economy")
EXTERNAL_COVARIATES_PATH = REPO_ROOT / "data" / "external_covariates"
TAG_RE = re.compile(r"[A-Za-z][A-Za-z0-9_+-]+")
EVENT_PATTERNS = {
    "election": r"election|president|prime minister|mayor|governor|parliament",
    "military": r"strike|war|missile|ceasefire|attack|troops|nuclear",
    "policy": r"tariff|fed|rate|ban|approval|regulation|sanction|etf",
    "corporate": r"earnings|revenue|ipo|acquisition|merger|bankruptcy",
    "crypto": r"bitcoin|btc|ethereum|eth|solana|crypto|token|airdrop",
}
CANDIDATE_PATTERNS = (
    re.compile(r"^will .+ be elected the next president of (.+)$"),
    re.compile(r"^will .+ win the (.+) election$"),
    re.compile(r"^which party will win (.+)$"),
)


def parse_listish(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return [chunk.strip() for chunk in text.split(",") if chunk.strip()]


def slug_family(slug: object) -> str:
    text = str(slug or "").lower()
    text = re.sub(r"-\d+(?:-\d+)+$", "", text)
    text = re.split(r"-(?:by|before|after|on|during|in)-", text, maxsplit=1)[0]
    text = re.sub(r"-(?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december).*", "", text)
    text = re.sub(r"-+$", "", text)
    return text or "unknown_family"


def candidate_family(question: object) -> str:
    q = re.sub(r"\s+", " ", str(question or "").lower()).strip(" ?")
    for pattern in CANDIDATE_PATTERNS:
        match = pattern.match(q)
        if match:
            return match.group(1).strip()
    return ""


def build_market_text(markets_df: pd.DataFrame) -> pd.DataFrame:
    work = markets_df.copy()
    work["tag_list"] = work["tag_labels"].apply(parse_listish)
    matched_domains = work["matched_domains"] if "matched_domains" in work.columns else pd.Series("", index=work.index)
    work["matched_domain_list"] = matched_domains.apply(parse_listish)
    work["question"] = work["question"].fillna("")
    work["description"] = work["description"].fillna("")
    work["resolution_source"] = work["resolution_source"].fillna("")
    work["market_text"] = (
        work["question"]
        + " [SEP] "
        + work["description"]
        + " [SEP] tags: "
        + work["tag_list"].apply(lambda values: " ".join(values))
        + " [SEP] source: "
        + work["resolution_source"]
    )
    work["family_id"] = work["market_slug"].apply(slug_family)
    work["candidate_family_id"] = work["question"].apply(candidate_family)
    work["tag_count"] = work["tag_list"].apply(len)
    work["matched_domain_count"] = work["matched_domain_list"].apply(len)
    work["question_char_len"] = work["question"].str.len()
    work["description_char_len"] = work["description"].str.len()
    work["has_resolution_source"] = work["resolution_source"].ne("").astype(int)
    for keyword, pattern in EVENT_PATTERNS.items():
        work[f"kw_{keyword}"] = work["market_text"].str.contains(pattern, case=False, regex=True).astype(int)

    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=2500)
    tfidf = vectorizer.fit_transform(work["market_text"])
    if tfidf.shape[1] >= 2 and len(work) >= 3:
        n_components = int(min(16, len(work) - 1, tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        embeddings = svd.fit_transform(tfidf)
        for idx in range(embeddings.shape[1]):
            work[f"text_svd_{idx:02d}"] = embeddings[:, idx]
        similarity = cosine_similarity(embeddings)
        np.fill_diagonal(similarity, -1.0)
        work["duplicate_neighbor_count"] = (similarity >= 0.82).sum(axis=1)
        work["max_text_similarity"] = np.where(len(work) > 1, similarity.max(axis=1), 0.0)
    else:
        work["duplicate_neighbor_count"] = 0
        work["max_text_similarity"] = 0.0

    family_stats = (
        work.groupby("family_id", dropna=False)
        .agg(
            family_market_count=("market_id", "size"),
            family_volume_sum=("volume_num", "sum"),
            family_end_date_span_days=("end_date", lambda s: (s.max() - s.min()).total_seconds() / 86400.0 if len(s) > 1 else 0.0),
        )
        .reset_index()
    )
    candidate_stats = (
        work.loc[work["candidate_family_id"].ne("")]
        .groupby("candidate_family_id", dropna=False)
        .agg(candidate_market_count=("market_id", "size"), candidate_volume_sum=("volume_num", "sum"))
        .reset_index()
    )
    work = work.merge(family_stats, on="family_id", how="left")
    work = work.merge(candidate_stats, on="candidate_family_id", how="left")
    work["candidate_market_count"] = work["candidate_market_count"].fillna(0)
    work["candidate_volume_sum"] = work["candidate_volume_sum"].fillna(0.0)
    return work


def load_multi_domain_markets(conn, *, domains, max_markets_per_domain, min_probability_rows):
    frames = []
    for domain in domains:
        frame = load_eligible_markets(
            conn,
            domain=domain,
            max_markets=max_markets_per_domain,
            min_probability_rows=min_probability_rows,
        )
        frames.append(frame)
    markets_df = pd.concat(frames, ignore_index=True)
    markets_df = markets_df.sort_values(["primary_domain", "volume_num", "created_at"], ascending=[True, False, False], kind="stable")
    markets_df = markets_df.drop_duplicates(subset=["market_id"], keep="first").reset_index(drop=True)
    return build_market_text(markets_df)


def build_relative_progress_terminal_dataset(
    markets_df: pd.DataFrame,
    probabilities_df: pd.DataFrame,
    *,
    progress_points: tuple[float, ...] = (0.25, 0.5, 0.75),
    max_snapshot_staleness_hours: float | None = 12.0,
) -> pd.DataFrame:
    grouped = {
        market_id: frame.reset_index(drop=True)
        for market_id, frame in probabilities_df.groupby("market_id", sort=False)
    }

    rows: list[dict[str, object]] = []
    for market in markets_df.itertuples(index=False):
        market_panel = grouped.get(str(market.market_id))
        if market_panel is None or market_panel.empty:
            continue

        total_lifetime = market.end_date - market.created_at
        if total_lifetime <= pd.Timedelta(0):
            continue

        for progress in progress_points:
            cutoff = market.created_at + progress * total_lifetime
            if cutoff <= market.created_at or cutoff >= market.end_date:
                continue

            features = extract_snapshot_features(
                market_panel,
                cutoff=cutoff,
                history_hours=(24, 24 * 7),
                max_snapshot_staleness_hours=max_snapshot_staleness_hours,
            )
            if features is None:
                continue

            observed_cutoff = pd.Timestamp(features["cutoff_timestamp_utc"])
            horizon_hours = (market.end_date - observed_cutoff).total_seconds() / 3600.0
            label = int(float(market.final_yes_probability) >= 0.5)
            base_prob = float(features["current_yes_probability"])
            rows.append(
                {
                    "market_id": str(market.market_id),
                    "market_slug": market.market_slug,
                    "question": market.question,
                    "end_date": market.end_date,
                    "created_at": market.created_at,
                    "volume_num": market.volume_num,
                    "liquidity_num": market.liquidity_num,
                    "trade_rows": market.trade_rows,
                    "probability_rows": market.probability_rows,
                    "horizon_hours": float(horizon_hours),
                    "horizon_name": f"{int(progress * 100)}%",
                    "horizon_family": "life_progress",
                    "snapshot_progress": float(progress),
                    "target": label,
                    "market_price_baseline": base_prob,
                    "market_abs_error": abs(label - base_prob),
                    "market_log_loss": log_loss([label], [np.clip(base_prob, 1e-6, 1 - 1e-6)], labels=[0, 1]),
                    **features,
                }
            )

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    return df.sort_values(["cutoff_timestamp_utc", "market_id", "horizon_name"], kind="stable").reset_index(drop=True)


def load_optional_covariates(path: Path):
    if not path.exists():
        return None
    covariates = load_external_covariates(path)
    if covariates.empty:
        return None
    value_col = "close" if "close" in covariates.columns else "value"
    wide = pivot_covariates_to_wide(covariates, value_col=value_col)
    features = add_lagged_covariate_features(wide, lags=(1, 12, 288), pct_change=True)
    return features


def join_covariates(base_df: pd.DataFrame, covariate_df: pd.DataFrame | None, *, time_col: str) -> pd.DataFrame:
    if covariate_df is None or covariate_df.empty:
        return base_df
    return asof_join_covariates(base_df, covariate_df, base_time_col=time_col, max_age="7D")


def add_manipulation_proxies(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6
    if "lookback_24h_trade_count_sum" in out.columns:
        trade_count = out["lookback_24h_trade_count_sum"].fillna(0.0)
        total_size = out["lookback_24h_total_size_sum"].fillna(0.0)
        volatility = out["lookback_24h_volatility"].fillna(0.0)
        abs_move = out["lookback_24h_abs_move_mean"].fillna(0.0)
        max_move = out["lookback_24h_abs_move_max"].fillna(0.0)
        directional = out["lookback_24h_yes_probability_change"].fillna(0.0)
        observed_share = out["lookback_24h_observed_trade_share"].fillna(0.0)
        staleness = out.get("snapshot_staleness_hours", pd.Series(0.0, index=out.index)).fillna(0.0)
    else:
        trade_count = out["trade_count_sum"].fillna(0.0)
        total_size = out["total_size_sum"].fillna(0.0)
        volatility = out["recent_volatility"].fillna(0.0)
        abs_move = out["recent_abs_move_mean"].fillna(0.0)
        max_move = out["recent_abs_move_max"].fillna(0.0)
        directional = out["recent_directional_move"].fillna(0.0)
        observed_share = out["observed_trade_share"].fillna(0.0)
        staleness = pd.Series(0.0, index=out.index)

    confidence = out["confidence_margin"].fillna(0.0)
    out["avg_trade_size_proxy"] = total_size / (trade_count + eps)
    out["move_per_trade_proxy"] = directional.abs() / (trade_count + eps)
    out["move_per_dollar_proxy"] = directional.abs() / (total_size + eps)
    out["burstiness_proxy"] = max_move / (abs_move + eps)
    out["wash_proxy"] = observed_share * trade_count / (directional.abs() + volatility + 1e-4)
    out["conviction_without_flow_proxy"] = confidence / (observed_share + 0.05)
    out["stale_conviction_proxy"] = confidence * (1.0 + staleness)
    out["volatility_gap_proxy"] = volatility / (observed_share + 0.05)
    return out


def latest_probability_before(panel: pd.DataFrame, cutoff: pd.Timestamp) -> float:
    history = panel.loc[panel["timestamp_utc"] <= cutoff]
    if history.empty:
        return float("nan")
    return float(history["yes_probability"].iloc[-1])


def attach_family_snapshot_features(dataset: pd.DataFrame, probabilities_df: pd.DataFrame, markets_df: pd.DataFrame, *, time_col: str, prob_col: str) -> pd.DataFrame:
    panels = {market_id: frame.reset_index(drop=True) for market_id, frame in probabilities_df.groupby("market_id", sort=False)}
    family_members = markets_df.groupby("family_id", dropna=False)["market_id"].agg(list).to_dict()
    candidate_members = (
        markets_df.loc[markets_df["candidate_family_id"].ne("")]
        .groupby("candidate_family_id", dropna=False)["market_id"].agg(list).to_dict()
    )
    family_map = markets_df.set_index("market_id")["family_id"].to_dict()
    candidate_map = markets_df.set_index("market_id")["candidate_family_id"].to_dict()
    end_date_map = markets_df.set_index("market_id")["end_date"].to_dict()

    rows = []
    for row in dataset[["market_id", time_col, prob_col]].itertuples(index=False):
        market_id = str(row.market_id)
        cutoff = pd.Timestamp(getattr(row, time_col))
        current_prob = float(getattr(row, prob_col))
        family_id = family_map.get(market_id, "unknown_family")
        sibling_probs = []
        earlier_probs = []
        later_probs = []
        for sibling_id in family_members.get(family_id, []):
            if sibling_id == market_id:
                continue
            sibling_prob = latest_probability_before(panels[sibling_id], cutoff) if sibling_id in panels else float("nan")
            if np.isnan(sibling_prob):
                continue
            sibling_probs.append(sibling_prob)
            if end_date_map.get(sibling_id) < end_date_map.get(market_id):
                earlier_probs.append(sibling_prob)
            elif end_date_map.get(sibling_id) > end_date_map.get(market_id):
                later_probs.append(sibling_prob)

        candidate_family_id = candidate_map.get(market_id, "")
        candidate_probs = []
        if candidate_family_id:
            for sibling_id in candidate_members.get(candidate_family_id, []):
                sibling_prob = latest_probability_before(panels[sibling_id], cutoff) if sibling_id in panels else float("nan")
                if not np.isnan(sibling_prob):
                    candidate_probs.append(sibling_prob)

        later_violation = max(0.0, current_prob - min(later_probs)) if later_probs else 0.0
        earlier_violation = max(0.0, max(earlier_probs) - current_prob) if earlier_probs else 0.0
        candidate_sum = float(np.sum(candidate_probs)) if candidate_probs else current_prob
        rows.append(
            {
                "market_id": market_id,
                time_col: cutoff,
                "family_snapshot_size": len(sibling_probs) + 1,
                "family_prob_mean": float(np.mean(sibling_probs)) if sibling_probs else current_prob,
                "family_prob_std": float(np.std(sibling_probs)) if sibling_probs else 0.0,
                "family_prob_gap": current_prob - (float(np.mean(sibling_probs)) if sibling_probs else current_prob),
                "family_coherence_gap": later_violation + earlier_violation,
                "family_future_monotone_violation": later_violation,
                "family_past_monotone_violation": earlier_violation,
                "candidate_prob_sum_gap": abs(candidate_sum - 1.0) if candidate_probs else 0.0,
            }
        )

    features = pd.DataFrame(rows)
    return dataset.merge(features, on=["market_id", time_col], how="left")


def add_domain_dummies(df: pd.DataFrame) -> pd.DataFrame:
    if "primary_domain" not in df.columns:
        return df
    dummies = pd.get_dummies(df["primary_domain"], prefix="domain", dtype=float)
    return pd.concat([df, dummies], axis=1)


def confidence_slice(prob: float) -> str:
    prob = float(prob)
    if prob <= 0.10 or prob >= 0.90:
        return "saturated (<=10% or >=90%)"
    if 0.25 <= prob <= 0.75:
        return "hard (25%-75%)"
    if 0.15 <= prob <= 0.85:
        return "medium (15%-85%)"
    return "confident (10%-15% or 85%-90%)"


def safe_auc(y_true, pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, pred)


def clipped(values, eps=1e-6):
    return np.clip(np.asarray(values, dtype=float), eps, 1.0 - eps)


def summarize_by_group(df: pd.DataFrame, by: list[str], metric_cols: list[str]) -> pd.DataFrame:
    return (
        df.groupby(by, dropna=False)[metric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values(by)
    )

## Dataset Construction

Here we build the benchmark dataset from raw tables.

Which tables are used:
- `markets`: market metadata and final outcome;
- `added_markets`: information about which markets were fully downloaded;
- `probabilities`: 5-minute `yes_probability` history.

Which filters are used:
- we only keep markets with sufficient history length;
- we require valid `created_at` and `end_date`;
- for each horizon, we only keep snapshots that truly exist and are not too stale;
- during training we do not use lifetime aggregates such as final total market volume, so we do not mix future information into the features.

**Unit of evaluation:**
- one row = one market at one fixed snapshot;
- the snapshot can be defined either by absolute time to resolution or by fraction of the market lifetime already elapsed.

In [2]:
MAX_MARKETS_PER_DOMAIN = 180
ABSOLUTE_HORIZONS_HOURS = (24, 72, 168)
PROGRESS_POINTS = (0.25, 0.50, 0.75)
MIN_PROBABILITY_ROWS = 24 * 12
MAX_SNAPSHOT_STALENESS_HOURS = 12
HORIZON_ORDER = ["25%", "50%", "75%", "-7d", "-3d", "-1d"]

conn = connect(DB_PATH)
markets_df = load_multi_domain_markets(
    conn,
    domains=DOMAINS,
    max_markets_per_domain=MAX_MARKETS_PER_DOMAIN,
    min_probability_rows=MIN_PROBABILITY_ROWS,
)
probabilities_df = load_probabilities_for_markets(conn, markets_df["market_id"].tolist())

absolute_terminal_df = build_multi_horizon_terminal_dataset(
    markets_df,
    probabilities_df,
    horizons_hours=ABSOLUTE_HORIZONS_HOURS,
    max_snapshot_staleness_hours=MAX_SNAPSHOT_STALENESS_HOURS,
)
absolute_terminal_df["horizon_family"] = "absolute_time"
absolute_terminal_df["snapshot_progress"] = np.nan
absolute_terminal_df["horizon_name"] = absolute_terminal_df["horizon_hours"].map({24: "-1d", 72: "-3d", 168: "-7d"}).fillna(absolute_terminal_df["horizon_name"])

progress_terminal_df = build_relative_progress_terminal_dataset(
    markets_df,
    probabilities_df,
    progress_points=PROGRESS_POINTS,
    max_snapshot_staleness_hours=MAX_SNAPSHOT_STALENESS_HOURS,
)

terminal_df = pd.concat([absolute_terminal_df, progress_terminal_df], ignore_index=True, sort=False)
terminal_df = add_time_features(terminal_df)
terminal_df["horizon_name"] = pd.Categorical(terminal_df["horizon_name"], categories=HORIZON_ORDER, ordered=True)
terminal_df["confidence_slice"] = terminal_df["current_yes_probability"].map(confidence_slice)
terminal_df["hard_case_10_90"] = terminal_df["current_yes_probability"].between(0.10, 0.90, inclusive="neither")
terminal_df["very_hard_case_25_75"] = terminal_df["current_yes_probability"].between(0.25, 0.75, inclusive="both")

market_feature_cols = [
    "primary_domain",
    "market_text",
    "family_id",
    "candidate_family_id",
    "tag_count",
    "matched_domain_count",
    "question_char_len",
    "description_char_len",
    "has_resolution_source",
    "duplicate_neighbor_count",
    "max_text_similarity",
    "family_market_count",
    "family_volume_sum",
    "family_end_date_span_days",
    "candidate_market_count",
    "candidate_volume_sum",
] + [
    col for col in markets_df.columns if col.startswith("kw_") or col.startswith("text_svd_")
]
terminal_df = terminal_df.merge(markets_df[["market_id", *market_feature_cols]], on="market_id", how="left")
terminal_df = attach_family_snapshot_features(
    terminal_df,
    probabilities_df,
    markets_df,
    time_col="cutoff_timestamp_utc",
    prob_col="current_yes_probability",
)
covariates_df = load_optional_covariates(EXTERNAL_COVARIATES_PATH)
terminal_df = join_covariates(terminal_df, covariates_df, time_col="cutoff_timestamp_utc")
terminal_df = add_manipulation_proxies(terminal_df)
terminal_df = add_domain_dummies(terminal_df)

history_df = markets_df[["market_id", "market_slug", "primary_domain", "probability_start_utc", "probability_end_utc", "probability_rows"]].copy()
history_df["history_duration_hours"] = (
    (history_df["probability_end_utc"] - history_df["probability_start_utc"]).dt.total_seconds() / 3600.0
)
history_df["history_duration_days"] = history_df["history_duration_hours"] / 24.0
terminal_df = terminal_df.merge(history_df[["market_id", "history_duration_hours", "history_duration_days"]], on="market_id", how="left")

print(f"markets: {len(markets_df):,}")
print(f"probability rows: {len(probabilities_df):,}")
print(f"terminal rows: {len(terminal_df):,}")

display(
    history_df.groupby("primary_domain", dropna=False)
    .agg(
        markets=("market_id", "nunique"),
        median_history_days=("history_duration_days", "median"),
        p25_history_days=("history_duration_days", lambda s: s.quantile(0.25)),
        p75_history_days=("history_duration_days", lambda s: s.quantile(0.75)),
        median_probability_rows=("probability_rows", "median"),
    )
    .reset_index()
)
display(
    history_df.sort_values("history_duration_days", ascending=False)
    [["market_slug", "primary_domain", "history_duration_days", "probability_rows"]]
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(
    data=history_df,
    x="history_duration_days",
    hue="primary_domain",
    bins=28,
    element="step",
    stat="count",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Продолжительность истории по тикерам")
axes[0].set_xlabel("Дней доступной 5m-истории")

sns.scatterplot(
    data=history_df,
    x="history_duration_days",
    y="probability_rows",
    hue="primary_domain",
    alpha=0.75,
    ax=axes[1],
)
axes[1].set_title("Длина истории vs число 5m-точек")
axes[1].set_xlabel("Дней доступной 5m-истории")
axes[1].set_ylabel("Число строк в probabilities")
plt.tight_layout()
plt.show()

display(
    terminal_df.groupby(["horizon_family", "horizon_name", "primary_domain"], dropna=False)
    .agg(
        rows=("target", "size"),
        positive_rate=("target", "mean"),
        baseline_log_loss=("market_log_loss", "mean"),
        hard_case_share=("hard_case_10_90", "mean"),
        family_gap=("family_coherence_gap", "mean"),
    )
    .reset_index()
    .sort_values(["horizon_family", "horizon_name", "rows"], ascending=[True, True, False])
)
display(
    terminal_df.groupby(["horizon_family", "confidence_slice"], dropna=False)
    .agg(rows=("target", "size"), baseline_log_loss=("market_log_loss", "mean"))
    .reset_index()
    .sort_values(["horizon_family", "rows"], ascending=[True, False])
)
display(terminal_df.head())

markets: 360
probability rows: 7,505,577
terminal rows: 1,692


## Evaluation Protocol and Baselines

**Split:**
- out-of-time rolling splits;
- train is always earlier than test by `cutoff_timestamp_utc`, not by `end_date`.

**Main metrics:**
- `log_loss` as the main probabilistic metric;
- `brier` as probability quality;
- `roc_auc` as a ranking-based sanity check.

**Leakage rules:**
- we cannot use data after the snapshot time;
- we cannot use post-resolution information;
- lifetime aggregates that know how much trading will happen later are excluded from the feature set;
- we separately analyze saturated and hard-case states so that already nearly resolved markets do not hide the real model behavior.

**Baselines:**
- `market_price` as the most important trivial baseline;
- a simple supervised linear model;
- a stronger tree / boosting baseline;
- additional ablations if we want to understand the contribution of microstructure, structure, text, or external covariates.

In [3]:
feature_family_map = {
    "text": [col for col in terminal_df.columns if col.startswith("text_svd_") or col.startswith("kw_") or col in {"tag_count", "matched_domain_count", "question_char_len", "description_char_len", "has_resolution_source", "duplicate_neighbor_count", "max_text_similarity"}],
    "graph": [col for col in terminal_df.columns if col.startswith("family_") or col.startswith("candidate_")],
    "external": [col for col in terminal_df.columns if col.startswith("btc_usd") or col.startswith("eth_usd")],
    "manipulation": [col for col in terminal_df.columns if col.endswith("_proxy")],
    "domain": [col for col in terminal_df.columns if col.startswith("domain_")],
}
feature_family_map["graph"] = [
    col for col in feature_family_map["graph"]
    if pd.api.types.is_numeric_dtype(terminal_df[col])
]
base_excluded = {
    "market_id", "market_slug", "question", "end_date", "created_at", "cutoff_timestamp_utc", "target", "market_text",
    "family_id", "candidate_family_id", "market_abs_error", "market_log_loss", "resolution_source", "tag_labels", "matched_domains",
    "volume_num", "liquidity_num", "trade_rows", "probability_rows", "history_duration_hours", "history_duration_days",
    "family_volume_sum", "candidate_volume_sum", "confidence_slice", "hard_case_10_90", "very_hard_case_25_75",
}
microstructure_cols = [
    col for col in terminal_df.columns
    if pd.api.types.is_numeric_dtype(terminal_df[col])
    and col not in base_excluded
    and col not in set().union(*feature_family_map.values())
]
full_feature_cols = sorted(set(microstructure_cols).union(*feature_family_map.values()))
text_graph_cols = sorted(set(microstructure_cols).union(feature_family_map["text"], feature_family_map["graph"], feature_family_map["domain"]))

logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2500)),
])
multimodal_hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(max_depth=5, learning_rate=0.04, max_iter=350, random_state=42)),
])
micro_hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=42)),
])


def append_metric_row(rows, *, fold, model, frame, pred, scope_type, scope_value, horizon_family=np.nan, horizon_name=np.nan, primary_domain=np.nan, confidence_slice_label=np.nan):
    if len(frame) < 10:
        return
    rows.append({
        "fold": fold,
        "model": model,
        "scope_type": scope_type,
        "scope_value": scope_value,
        "horizon_family": horizon_family,
        "horizon_name": horizon_name,
        "primary_domain": primary_domain,
        "confidence_slice": confidence_slice_label,
        "rows": int(len(frame)),
        "log_loss": float(log_loss(frame["target"], pred, labels=[0, 1])),
        "brier": float(brier_score_loss(frame["target"], pred)),
        "roc_auc": float(safe_auc(frame["target"], pred)),
    })


rows = []
for train_df, test_df, meta in rolling_time_splits(terminal_df, time_col="cutoff_timestamp_utc", n_splits=4, min_train_fraction=0.5):
    market_pred = clipped(test_df["market_price_baseline"])

    micro_hgb.fit(train_df[microstructure_cols], train_df["target"])
    micro_pred = clipped(micro_hgb.predict_proba(test_df[microstructure_cols])[:, 1])

    logistic.fit(train_df[text_graph_cols], train_df["target"])
    text_graph_pred = clipped(logistic.predict_proba(test_df[text_graph_cols])[:, 1])

    multimodal_hgb.fit(train_df[full_feature_cols], train_df["target"])
    full_pred = clipped(multimodal_hgb.predict_proba(test_df[full_feature_cols])[:, 1])

    pred_map = {
        "market_price": market_pred,
        "microstructure_hgb": micro_pred,
        "text_graph_logistic": text_graph_pred,
        "multimodal_hgb": full_pred,
    }
    for model_name, pred in pred_map.items():
        fold_view = test_df.assign(_pred=pred)
        append_metric_row(rows, fold=meta["fold"], model=model_name, frame=fold_view, pred=fold_view["_pred"], scope_type="overall", scope_value="overall")

        for (horizon_family, horizon_name), frame in fold_view.groupby(["horizon_family", "horizon_name"], dropna=False):
            append_metric_row(
                rows,
                fold=meta["fold"],
                model=model_name,
                frame=frame,
                pred=frame["_pred"],
                scope_type="horizon",
                scope_value=f"{horizon_family}|{horizon_name}",
                horizon_family=horizon_family,
                horizon_name=str(horizon_name),
            )

        for (horizon_family, horizon_name, primary_domain), frame in fold_view.groupby(["horizon_family", "horizon_name", "primary_domain"], dropna=False):
            append_metric_row(
                rows,
                fold=meta["fold"],
                model=model_name,
                frame=frame,
                pred=frame["_pred"],
                scope_type="horizon_domain",
                scope_value=f"{horizon_family}|{horizon_name}|{primary_domain}",
                horizon_family=horizon_family,
                horizon_name=str(horizon_name),
                primary_domain=primary_domain,
            )

        for confidence_slice_label, frame in fold_view.groupby("confidence_slice", dropna=False):
            append_metric_row(
                rows,
                fold=meta["fold"],
                model=model_name,
                frame=frame,
                pred=frame["_pred"],
                scope_type="confidence_slice",
                scope_value=str(confidence_slice_label),
                confidence_slice_label=str(confidence_slice_label),
            )

        hard_frame = fold_view.loc[fold_view["hard_case_10_90"]]
        append_metric_row(
            rows,
            fold=meta["fold"],
            model=model_name,
            frame=hard_frame,
            pred=hard_frame["_pred"],
            scope_type="hard_case",
            scope_value="0.1 < p < 0.9",
            confidence_slice_label="hard_case_10_90",
        )

terminal_metrics = pd.DataFrame(rows)
overall_summary = (
    terminal_metrics.loc[terminal_metrics["scope_type"].eq("overall")]
    .groupby("model")[["log_loss", "brier", "roc_auc"]]
    .mean()
    .sort_values("log_loss")
    .reset_index()
)
horizon_summary = (
    terminal_metrics.loc[terminal_metrics["scope_type"].eq("horizon")]
    .groupby(["horizon_family", "horizon_name", "model"])[["log_loss", "brier", "roc_auc"]]
    .mean()
    .reset_index()
    .sort_values(["horizon_family", "horizon_name", "log_loss"])
)
confidence_summary = (
    terminal_metrics.loc[terminal_metrics["scope_type"].eq("confidence_slice")]
    .groupby(["confidence_slice", "model"])[["log_loss", "brier", "roc_auc"]]
    .mean()
    .reset_index()
    .sort_values(["confidence_slice", "log_loss"])
)
hard_case_summary = (
    terminal_metrics.loc[terminal_metrics["scope_type"].eq("hard_case")]
    .groupby("model")[["log_loss", "brier", "roc_auc"]]
    .mean()
    .sort_values("log_loss")
    .reset_index()
)

display(overall_summary)
display(horizon_summary)
display(confidence_summary)
display(hard_case_summary)

## Results

This block contains the first benchmark results.

How to read them:
- first inspect the overall table across all folds;
- then inspect the breakdowns across absolute and life-progress horizons;
- then look at hard-case and confidence slices;
- after that compare the full model with simpler ablation baselines.

The main question is: **what improves over `market_price`, and where exactly does that improvement appear: early in the market lifecycle, closer to resolution, or only on difficult non-saturated states?**

In [4]:
ablation_specs = {
    "microstructure_only": microstructure_cols,
    "micro_plus_text_graph": text_graph_cols,
    "micro_plus_text_graph_external": sorted(set(text_graph_cols).union(feature_family_map["external"])),
    "full_multimodal": full_feature_cols,
}

ablation_rows = []
for name, cols in ablation_specs.items():
    fold_metrics = []
    for train_df, test_df, _ in rolling_time_splits(terminal_df, time_col="cutoff_timestamp_utc", n_splits=4, min_train_fraction=0.5):
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(max_depth=5, learning_rate=0.04, max_iter=350, random_state=42)),
        ])
        model.fit(train_df[cols], train_df["target"])
        pred = clipped(model.predict_proba(test_df[cols])[:, 1])
        fold_metrics.append({
            "log_loss": float(log_loss(test_df["target"], pred, labels=[0, 1])),
            "brier": float(brier_score_loss(test_df["target"], pred)),
            "roc_auc": float(safe_auc(test_df["target"], pred)),
        })
    summary = pd.DataFrame(fold_metrics).mean(numeric_only=True).to_dict()
    summary["spec"] = name
    ablation_rows.append(summary)

ablation_df = pd.DataFrame(ablation_rows).sort_values("log_loss")
display(ablation_df)

plot_df = terminal_metrics.loc[terminal_metrics["scope_type"].eq("overall")].groupby("model")[["log_loss", "roc_auc"]].mean().reset_index()
horizon_plot_df = (
    terminal_metrics.loc[terminal_metrics["scope_type"].eq("horizon")]
    .groupby(["horizon_family", "horizon_name", "model"])[["log_loss"]]
    .mean()
    .reset_index()
)
confidence_plot_df = (
    terminal_metrics.loc[terminal_metrics["scope_type"].eq("confidence_slice")]
    .groupby(["confidence_slice", "model"])[["log_loss"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.barplot(data=plot_df, x="model", y="log_loss", ax=axes[0], palette="crest")
axes[0].set_title("Overall log loss")
axes[0].tick_params(axis="x", rotation=25)

sns.lineplot(
    data=horizon_plot_df,
    x="horizon_name",
    y="log_loss",
    hue="model",
    style="horizon_family",
    markers=True,
    dashes=False,
    ax=axes[1],
)
axes[1].set_title("Log loss по горизонтам")
axes[1].set_xlabel("Горизонт")
axes[1].tick_params(axis="x", rotation=25)

sns.barplot(
    data=confidence_plot_df,
    x="confidence_slice",
    y="log_loss",
    hue="model",
    ax=axes[2],
)
axes[2].set_title("Log loss по confidence slices")
axes[2].tick_params(axis="x", rotation=30)
axes[2].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

/var/folders/zv/mjsbg2dx4q79c1gbwtlhbk180000gp/T/ipykernel_37781/2664569445.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=plot_df, x="model", y="log_loss", ax=axes[0], palette="crest")


## Interpretation

A good result here does not simply mean “we slightly beat the current market price.”

A strong paper story looks like this:
- the model uses market traces to better recover the final event probability;
- the improvement appears not only on one horizon, but at least on part of the temporal regimes;
- the gain is especially visible on hard-case states, where `market_price` is not yet nearly deterministic;
- the gain can be linked to a concrete type of information: microstructure, related-market structure, text, or external covariates.

If `market_price` remains the best baseline, that is still a useful scientific result:
- it means the prediction market is already a very strong information aggregator;
- further improvements should then be sought not in naive feature expansion, but in trust, coherence, or gated multimodal updates.